# Find Quads Benchmark

In [21]:
import time
from itertools import combinations
from collections import namedtuple
from bisect import bisect_left, bisect_right
import random

In [22]:
Peak = namedtuple("Peak", ["x", "y"])
Quad = namedtuple("Quad", ["A", "C", "D", "B"])

## Original Find Quads (combinations)

In [23]:
# def _valid_quad_combinations(q):
#     """
#     Evaluates:

#           Ay < By
#       Ax < Cx <= Dx <= Bx
#       Ay < Cy ,  Dy <= By

#     !! NOTE: assumes combinations are sorted by x value
#     (default behavior of itertools.combinations)
#     """
#     if q.A.y < q.C.y < q.B.y and q.A.y < q.D.y <= q.B.y:
#         return True
#     else:
#         return False

def _valid_quad_combinations(q):
    """
    Mantido para compatibilidade, mas a construção já garante as restrições.
    """
    return (q.A.y < q.B.y
            and q.A.x < q.C.x <= q.D.x <= q.B.x
            and q.A.y < q.C.y and q.D.y <= q.B.y)


In [24]:
def _valid_quads(root, filtered):
    """
    returns list of validated quads for given root (A)
    """
    Quad = namedtuple('Quad', ['A', 'C', 'D', 'B'])
    validQuads = []
    for comb in combinations(filtered, 3):
        quad = Quad(root, comb[0], comb[1], comb[2])
        if _valid_quad_combinations(quad):
            validQuads.append(quad)
    if len(validQuads) == 0:
        return None
    else:
        return validQuads

In [25]:
def _filter_peaks(root, peaks, r, c):
    """
    returns peaks inside window of Ax + c ± (r / 2)
    """
    lastPeak = peaks[-1].x
    windowStart = root.x + c - (r / 2)
    if windowStart > lastPeak:
        return None
    windowEnd = windowStart + r
    idx_start = bisect_left(peaks, (windowStart, 0))
    idx_end = bisect_right(peaks, (windowEnd, 0))
    filtered = peaks[idx_start:idx_end]
    if len(filtered) < 3:
        return None
    return filtered

In [26]:
def _root_quads_combinations(root, peaks, r, c):
    """
    finds valid quads for given root
    """
    quads = []
    filtered = _filter_peaks(root, peaks, r, c)
    if filtered is None:
        return []
    found = _valid_quads(root, filtered)
    if found is not None:
        quads += found
    return quads

In [27]:
def find_quads_combinations(peaks, r, c):
    """
    Returns list of valid/strong quads for list of peaks
    """
    quads = []
    for root in peaks:
        quads += _root_quads_combinations(root, peaks, r, c)
    return quads

## Segment Tree Find Quads

In [28]:
# class SegmentTree:
#     def __init__(self, points):
#         self.points = points
#         self.n = len(points)
#         self.tree = [[] for _ in range(4 * self.n)]
#         self._build(1, 0, self.n - 1)

#     def _build(self, node, l, r):
#         if l == r:
#             p = self.points[l]
#             self.tree[node] = [p]
#         else:
#             mid = (l + r) // 2
#             self._build(2 * node, l, mid)
#             self._build(2 * node + 1, mid + 1, r)
#             self.tree[node] = self.tree[2*node] + self.tree[2*node+1]
#             self.tree[node].sort(key=lambda p: p.y)  # ordena por y para bisect

#     def query(self, node, l, r, ql, qr, Dy_min):
#         if qr < l or ql > r:
#             return []
#         if ql <= l and r <= qr:
#             arr = self.tree[node]
#             from bisect import bisect_left
#             idx = bisect_left([p.y for p in arr], Dy_min)
#             return arr[idx:]
#         mid = (l + r) // 2
#         return self.query(2*node, l, mid, ql, qr, Dy_min) + \
#                self.query(2*node+1, mid+1, r, ql, qr, Dy_min)
    
#     def query_all(self):
#         """Retorna todos os pontos, podendo futuramente aplicar filtro rápido"""
#         return self.points

In [29]:
# def find_quads_seg_tree(peaks, r, c):
#     """
#     Retorna lista de quads válidos a partir dos picos
#     """
#     quads = []
#     for root in peaks:
#         quads += _root_quads_seg_tree(root, peaks, r, c)
#     return quads


# def _root_quads_seg_tree(root, peaks, r, c):
#     """
#     Gera quads para um pico root
#     """
#     filtered = _filter_peaks_seg_tree(root, peaks, r, c)
#     if filtered is None:
#         return []
#     filtered.sort(key=lambda p: p.x)
#     # Construímos a Segment Tree para esse subconjunto
#     segtree = SegmentTree(filtered)

#     found = _valid_quads_seg_tree(root, filtered, segtree)
#     return found if found else []


# def _filter_peaks_seg_tree(root, peaks, r, c):
#     """
#     Seleciona picos na janela de Ax + c ± (r / 2)
#     """
#     lastPeak = peaks[-1].x
#     windowStart = root.x + c - (r / 2)
#     if windowStart > lastPeak:
#         return None
#     windowEnd = windowStart + r
#     idx_start = bisect_left(peaks, (windowStart, 0))
#     idx_end = bisect_right(peaks, (windowEnd, 0))
#     filtered = peaks[idx_start:idx_end]
#     if len(filtered) < 3:
#         return None
#     return filtered


# def _valid_quads_seg_tree(root, filtered, segtree):
#     """
#     Gera quads válidos usando árvore de combinações + segment tree
#     """
#     Quad = namedtuple('Quad', ['A', 'C', 'D', 'B'])
#     validQuads = []

#     # Nível 1: escolha de C
#     for C in filtered:
#         if C == root:
#             continue
#         if not (C.x > root.x and C.y > root.y):
#             continue  # Ay < Cy, Ax < Cx

#         # Nível 2: escolha de D
#         for D in filtered:
#             if D in (root, C):
#                 continue
#             if not (D.x >= C.x):
#                 continue  # Ax < Cx <= Dx

#             # Nível 3: escolha de B via segment tree
#             # Consulta todos os pontos B com Bx >= Dx
#             B_candidates = segtree.query_all()  # todos os pontos
#             for B in B_candidates:
#                 if B in (root, C, D):
#                     continue
#                 if B.x < D.x:
#                     continue  # Dx <= Bx

#                 # Verificação completa das desigualdades
#                 if (root.y < B.y and  # Ay < By
#                     D.y <= B.y and     # Dy <= By
#                     C.y > root.y and   # Ay < Cy (já garantido)
#                     root.x < C.x <= D.x <= B.x):
#                     validQuads.append(Quad(root, C, D, B))

#     return validQuads if validQuads else None


segment tree yield

In [30]:
def find_quads_seg_tree(peaks, r, c):
    """
    Retorna lista de quads válidos a partir dos picos
    """
    for root in peaks:
        yield from _root_quads_seg_tree(root, peaks, r, c)



def _root_quads_seg_tree(root, peaks, r, c):
    """
    Gera quads para um pico root
    """
    filtered = _filter_peaks_seg_tree(root, peaks, r, c)
    if not filtered:
        return

    segtree = SegmentTree(filtered)  # ainda serve de interface
    yield from _valid_quads_seg_tree(root, filtered, segtree)


def _filter_peaks_seg_tree(root, peaks, r, c):
    """
    Seleciona picos na janela de Ax + c ± (r / 2)
    """
    lastPeak = peaks[-1].x
    windowStart = root.x + c - (r / 2)
    if windowStart > lastPeak:
        return None
    windowEnd = windowStart + r
    idx_start = bisect_left(peaks, (windowStart, 0))
    idx_end = bisect_right(peaks, (windowEnd, 0))
    filtered = peaks[idx_start:idx_end]
    if len(filtered) < 3:
        return None
    return sorted(filtered, key=lambda p: p.x)  # garante ordenação


def _valid_quads_seg_tree(root, filtered, segtree):
    """
    Gera quads válidos usando árvore de combinações + segment tree
    """
    Quad = namedtuple('Quad', ['A', 'C', 'D', 'B'])
    for C in filtered:
        if C == root or not (C.x > root.x and C.y > root.y):
            continue

        for D in filtered:
            if D in (root, C) or D.x < C.x:
                continue

            for B in segtree.query_all():
                if B in (root, C, D) or B.x < D.x:
                    continue

                if (root.y < B.y and
                    D.y <= B.y and
                    root.x < C.x <= D.x <= B.x):
                    yield Quad(root, C, D, B)



class SegmentTree:
    def __init__(self, points):
        self.points = points

    def query_all(self):
        return self.points


In [31]:
Peak = namedtuple("Peak", ["x", "y"])
Quad = namedtuple("Quad", ["A", "C", "D", "B"])

def find_quads_efficient(peaks, r, c):
    # Ordena globalmente por x (uma vez só)
    peaks_sorted = sorted(peaks, key=lambda p: p.x)
    xs = [p.x for p in peaks_sorted]

    for root in peaks_sorted:
        window_start = root.x + c - (r / 2)
        window_end = window_start + r

        s = bisect_left(xs, window_start)
        e = bisect_right(xs, window_end)
        if e - s < 3:
            continue

        # candidatos na janela, sem o próprio root
        filtered = [p for p in peaks_sorted[s:e] if p is not root]
        n = len(filtered)
        if n < 3:
            continue

        # gera C
        for i, C in enumerate(filtered):
            if C.x <= root.x or C.y <= root.y:
                continue  # precisa estar acima e à direita

            # gera D
            for j in range(i + 1, n):
                D = filtered[j]
                if D.x < C.x:
                    continue  # precisa estar à direita de C

                y_min = max(root.y + 1, D.y)

                # gera B
                for k in range(j + 1, n):
                    B = filtered[k]
                    if B.x < D.x or B.y < y_min:
                        continue
                    yield Quad(root, C, D, B)

## Funcionando

In [32]:
def find_quads_tree(peaks, r, c):
    """
    Returns list of valid/strong quads for list of peaks
    """
    quads = []
    for root in peaks:
        quads += _root_quads_tree(root, peaks, r, c)
    return quads


def _root_quads_tree(root, peaks, r, c):
    """
    finds valid quads for given root
    """
    filtered = _filter_peaks_tree(root, peaks, r, c)
    if filtered is None:
        return []
    return _build_quads_tree(root, filtered)


def _filter_peaks_tree(root, peaks, r, c):
    """
    returns peaks inside window of Ax + c ± (r / 2)
    """
    lastPeak = peaks[-1].x
    windowStart = root.x + c - (r / 2)
    if windowStart > lastPeak:
        return None
    windowEnd = windowStart + r
    idx_start = bisect_left(peaks, (windowStart, 0))
    idx_end = bisect_right(peaks, (windowEnd, 0))
    filtered = peaks[idx_start:idx_end]
    if len(filtered) < 3:
        return None
    return filtered


def _build_quads_tree(A, candidates):
    """
    Gera quads válidos via árvore de busca:
    - Nível 1: escolhe C (Ax < Cx, Ay < Cy)
    - Nível 2: escolhe D (Cx <= Dx, Dy <= By)
    - Nível 3: escolhe B (Bx > Ax, By > Ay)
    """
    quads = []
    n = len(candidates)

    # Percorrer candidatos a B
    for b_idx in range(n):
        B = candidates[b_idx]
        if not (A.y < B.y and B.x > A.x):
            continue

        # Percorrer candidatos a C
        for c_idx in range(b_idx):  # C deve vir antes de B no tempo
            C = candidates[c_idx]
            if not (A.x < C.x < B.x and A.y < C.y < B.y):
                continue

            # Percorrer candidatos a D
            for d_idx in range(c_idx, b_idx):  # D entre C e B
                D = candidates[d_idx]
                if not (C.x <= D.x <= B.x and D.y <= B.y):
                    continue

                quads.append(Quad(A, C, D, B))

    return quads

## Benchmark

In [33]:
random.seed(0)
peaks = [Peak(x, random.randint(1,50)) for x in range(0,100,10)]
r, c = 100, 20  # janela

# Original (combinations)
t0 = time.time()
quads1 = find_quads_combinations(peaks, r, c)
t1 = time.time()

# Segment Tree
t2 = time.time()
quads2 = list(find_quads_seg_tree(peaks, r, c))
t3 = time.time()

# Find quads lazy
t4 = time.time()
quads3 = list(find_quads_efficient(peaks, r, c))
t5 = time.time()

# Comparação de resultados
set1 = set(quads1)
set2 = set(quads2)

print(f"Combinations: {len(quads1)} quads, tempo {t1-t0:.4f}s")
print(f"SegmentTree : {len(quads2)} quads, tempo {t3-t2:.4f}s")
print(f"Lazy : {len(quads3)} quads, tempo {t5-t4:.4f}s")
print("Resultados iguais? ", set1 == set2)
print("Quads combinations: ", quads1)
# print("Quads segment tree: ", quads2)
print("Quads lazy:         ", quads3)

Combinations: 22 quads, tempo 0.0015s
SegmentTree : 22 quads, tempo 0.0007s
Lazy : 30 quads, tempo 0.0001s
Resultados iguais?  True
Quads combinations:  [Quad(A=Peak(x=0, y=25), C=Peak(x=10, y=49), D=Peak(x=20, y=27), B=Peak(x=50, y=33)), Quad(A=Peak(x=0, y=25), C=Peak(x=10, y=49), D=Peak(x=20, y=27), B=Peak(x=60, y=32)), Quad(A=Peak(x=0, y=25), C=Peak(x=10, y=49), D=Peak(x=30, y=3), B=Peak(x=50, y=33)), Quad(A=Peak(x=0, y=25), C=Peak(x=10, y=49), D=Peak(x=30, y=3), B=Peak(x=60, y=32)), Quad(A=Peak(x=0, y=25), C=Peak(x=10, y=49), D=Peak(x=40, y=17), B=Peak(x=50, y=33)), Quad(A=Peak(x=0, y=25), C=Peak(x=10, y=49), D=Peak(x=40, y=17), B=Peak(x=60, y=32)), Quad(A=Peak(x=0, y=25), C=Peak(x=20, y=27), D=Peak(x=30, y=3), B=Peak(x=50, y=33)), Quad(A=Peak(x=0, y=25), C=Peak(x=20, y=27), D=Peak(x=30, y=3), B=Peak(x=60, y=32)), Quad(A=Peak(x=0, y=25), C=Peak(x=20, y=27), D=Peak(x=40, y=17), B=Peak(x=50, y=33)), Quad(A=Peak(x=0, y=25), C=Peak(x=20, y=27), D=Peak(x=40, y=17), B=Peak(x=60, y=32)), 

In [34]:
def encontrar_quads_duplicados(quads):
    vistos = set()
    duplicados = set()
    
    for quad in quads:
        if quad in vistos:
            duplicados.add(quad)
        else:
            vistos.add(quad)
    
    return list(duplicados)

In [35]:
encontrar_quads_duplicados(quads1)

[]

In [36]:
encontrar_quads_duplicados(quads2)

[]

In [37]:
def check_quads_constraints(quads):
    """
    Verifica se todos os quads em `quads` obedecem às restrições:

        (1a) Ay < By
        (1b) Ax < Cx <= Dx <= Bx
        (1c) Ay < Cy , Dy <= By

    Retorna:
        - True se todos os quads são válidos
        - False caso algum quad viole as restrições
        - Além disso, imprime os quads inválidos encontrados
    """
    valid = True
    for q in quads:
        cond1 = q.A.y < q.B.y
        cond2 = (q.A.x < q.C.x <= q.D.x <= q.B.x)
        cond3 = (q.A.y < q.C.y) and (q.D.y <= q.B.y)

        if not (cond1 and cond2 and cond3):
            print("Quad inválido:", q)
            print(f"  cond1(Ay<By): {cond1}, cond2(Ax<Cx<=Dx<=Bx): {cond2}, cond3(y's): {cond3}")
            valid = False
    return valid


In [38]:
check_quads_constraints(quads1)

True

In [39]:
check_quads_constraints(quads2)

True

In [40]:
set1 = set((q.A.x, q.A.y, q.C.x, q.C.y, q.D.x, q.D.y, q.B.x, q.B.y) for q in quads1)
set2 = set((q.A.x, q.A.y, q.C.x, q.C.y, q.D.x, q.D.y, q.B.x, q.B.y) for q in quads2)

missing_in_tree = set1 - set2
extra_in_tree   = set2 - set1

print("Total combinations:", len(set1))
print("Total tree:", len(set2))
print("Missing in tree:", len(missing_in_tree))
print("Extra in tree:", len(extra_in_tree))

# Opcional: imprimir alguns exemplos para inspecionar
print("Exemplo faltando:", list(missing_in_tree)[:5])
print("Exemplo sobrando:", list(extra_in_tree)[:5])


Total combinations: 22
Total tree: 22
Missing in tree: 0
Extra in tree: 0
Exemplo faltando: []
Exemplo sobrando: []
